# Accessing Data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Holiday Function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

In [ ]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Relative Ranking
- for every hour (24 hour period) rank each data point relative to the others on the amount of demand used
- produces a list of 1 - 122, this ranking is done for each hour
- relative ts = [56, 74, 2, ...]
- day 56 out of the list (1-122) had the highest electricity demand at 12pm, day 74 had the highest demand at 12pm etc etc
- christmas = day 30
- at 12pm, day 30 ranked 12 : relative rank = n/122 (12/122 = 0.1) therefore at 12pm christmas day electricity was in the lowest 10% (at 12pm on christmas day electricity demand was lower relative another day in those 122 days)

## Downloading libraries

In [ ]:
import os
import pandas as pdf
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from dateutil.easter import easter

## Function: Public Holiday Relative Ranking Plot

In [ ]:
def holiday_relative_rank_plot(demand, info, station, years, holiday_func, holiday_name):
    """
    Computes the hour-stratified relative ranking of electricity demand
    for a specific public holiday across multiple years.
    Produces a 24-hour curve of relative rank (0 = lowest, 1 = highest).
    """

    # Ensure datetime index
    demand.index = pd.to_datetime(demand.index)

    # Hourly mean demand
    hourly = demand[[station]].resample("h").mean()

    # 30-day window around holiday for each year
    windows = []
    for year in years:
        ref_date = holiday_func(year)
        start = ref_date - pd.Timedelta(days=30)
        end   = ref_date + pd.Timedelta(days=30)
        win = hourly.loc[start:end].copy()
        windows.append(win)

    combined = pd.concat(windows)
    combined["date"] = combined.index.date
    combined["hour"] = combined.index.hour

    # --- Compute relative rank for EACH hour ---
    combined["rank"] = combined.groupby("hour")[station].rank(method="average")

    # Convert to percentile
    n_days = combined.groupby("hour")["date"].transform("nunique")
    combined["relative_rank"] = combined["rank"] / n_days

    # --- Extract holiday profile (mean across years) ---
    holiday_hours = []
    for year in years:
        ref_date = holiday_func(year)
        expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")

        daily = combined["relative_rank"].reindex(expected_hours)
        daily.index = range(24)
        holiday_hours.append(daily)

    holiday_profile = pd.concat(holiday_hours, axis=1).mean(axis=1)

    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(10,4))

    ax.plot(
        holiday_profile.index,
        holiday_profile.values,
        color="red",
        marker="o",
        linewidth=2,
        label=f"{holiday_name}"
    )

    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_yticklabels(["Lowest", "25%", "50%", "75%", "Highest"])

    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(f"{full_name} Relative Demand Rank: {holiday_name} ({years[0]}–{years[-1]})")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Relative Rank (0 = lowest, 1 = highest)")
    ax.legend()

    return fig

In [ ]:
station = "BLAKE"
years = [2015, 2016]
holiday_name = "Christmas Day"
holiday_func = HOLIDAYS_VIC[holiday_name]

fig = holiday_relative_rank_plot(demand, info, station, years, holiday_func, holiday_name)
plt.show()

## Christmas relative lines computed only relative to Christmas Day
- not this

In [ ]:
def christmas_relative_rank_all_years(demand, info, station, years):
    """
    Computes the hour‑stratified relative ranking of electricity demand
    for Christmas Day across multiple years.
    Plots one line per Christmas Day (2004–2017).
    """

    demand.index = pd.to_datetime(demand.index)
    hourly = demand[[station]].resample("h").mean()

    # --- Collect all Christmas Days ---
    daily_profiles = []
    labels = []

    for year in years:
        ref_date = pd.Timestamp(f"{year}-12-25")
        day_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")

        # Extract the 24 hours for that Christmas
        df_day = hourly.reindex(day_hours)

        # Skip if missing data
        if df_day[station].isna().all():
            continue

        df_day["hour"] = df_day.index.hour
        df_day["date"] = df_day.index.date
        daily_profiles.append(df_day)
        labels.append(year)

    combined = pd.concat(daily_profiles)

    # --- Compute relative rank within each hour across ALL Christmas Days ---
    combined["rank"] = combined.groupby("hour")[station].rank(method="average")
    n_days = combined.groupby("hour")["date"].transform("nunique")
    combined["relative_rank"] = combined["rank"] / n_days

    # --- Build one curve per Christmas Day ---
    curves = (
        combined.groupby(["date", "hour"])["relative_rank"]
        .mean()
        .unstack(level="hour")
    )

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10,4))

    # Pastel colour map
    import matplotlib
    cmap = matplotlib.colormaps.get_cmap("Pastel1").resampled(len(curves))

    # 1) Plot each Christmas Day as a light line
    for i, (day, row) in enumerate(curves.iterrows()):
        ax.plot(
            row.index,
            row.values,
            color=cmap(i),
            linewidth=1.5,
            alpha=0.8,
            label=str(day.year)
        )

    # 2) Add mean Christmas profile (optional)
    #mean_profile = curves.mean(axis=0)
    #ax.plot(
        #mean_profile.index,
        #mean_profile.values,
        #color="red",
        #linewidth=3,
        #marker="o",
        #label="Mean Christmas"
    #)

    # Formatting
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_yticklabels(["Lowest", "25%", "50%", "75%", "Highest"])

    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(f"{full_name} Relative Demand Rank: Christmas Day (2004–2017)")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Relative Rank (0 = lowest, 1 = highest)")
    ax.legend(ncol=3, fontsize=8)

    return fig

In [ ]:
station = "BLAKE"
years = list(range(2004, 2018))

fig = christmas_relative_rank_all_years(demand, info, station, years)
plt.close()

## Christmas only lines but relative to the 30 +/- days around the holiday for that year
- builds a ±30‑day window
- Compute relative ranks within that window
- Extract the Christmas Day curve
- Plot it as a light pastel line
- Add a bold red mean Christmas curve across all years
- multi‑year comparison of Christmas behaviour, but each relative to the 60 days surrounding christmas day

In [ ]:
def christmas_relative_rank_all_years_window(demand, info, station, years):
    """
    For each Christmas Day in the given years:
        • Build a ±30-day window around Christmas
        • Compute relative ranks within each hour using ALL days in that window
        • Extract the 24-hour Christmas Day relative-rank curve
    Plot:
        • One line per Christmas Day (light colours)
        • Optional mean Christmas curve (bold red)
    """

    demand.index = pd.to_datetime(demand.index)
    hourly = demand[[station]].resample("h").mean()

    christmas_curves = []   # store each year's 24-hour curve
    christmas_labels = []   # store year labels

    for year in years:
        # --- Build ±30-day window ---
        ref_date = pd.Timestamp(f"{year}-12-25")
        start = ref_date - pd.Timedelta(days=30)
        end   = ref_date + pd.Timedelta(days=30)

        window = hourly.loc[start:end].copy()
        if window.empty:
            continue

        window["date"] = window.index.date
        window["hour"] = window.index.hour

        # --- Compute relative rank within this window ---
        window["rank"] = window.groupby("hour")[station].rank(method="average")
        n_days = window.groupby("hour")["date"].transform("nunique")
        window["relative_rank"] = window["rank"] / n_days

        # --- Extract Christmas Day 24-hour curve ---
        expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")
        christmas_day = window["relative_rank"].reindex(expected_hours)

        # Skip if missing data
        if christmas_day.isna().all():
            continue

        christmas_day.index = range(24)
        christmas_curves.append(christmas_day)
        christmas_labels.append(year)

    # Convert to DataFrame: rows = years, columns = hours
    curves_df = pd.DataFrame(christmas_curves, index=christmas_labels)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10,4))

    # Pastel colour map
    import matplotlib
    cmap = matplotlib.colormaps.get_cmap("tab10").resampled(len(curves_df))

    # 1) Plot each Christmas Day as a light line
    for i, (year, row) in enumerate(curves_df.iterrows()):
        ax.plot(
            row.index,
            row.values,
            color=cmap(i),
            linewidth=1.5,
            alpha=0.9,
            label=str(year)
        )

    # 2) Mean Christmas curve (optional)
    #mean_curve = curves_df.mean(axis=0)
    #ax.plot(
        #mean_curve.index,
        #mean_curve.values,
        #color="red",
        #linewidth=2,
        #marker="o",
        #label="Mean Christmas"
    #)

    # Formatting
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_yticklabels(["Lowest", "25%", "50%", "75%", "Highest"])

    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(f"{full_name} Relative Demand Rank: Christmas Days (±30d window, {years[0]}–{years[-1]})")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Relative Rank (0 = lowest, 1 = highest)")
    ax.legend(ncol=3, fontsize=8)

    return fig

In [ ]:
station = "BLAKE"
years = list(range(2004, 2018))

fig = christmas_relative_rank_all_years_window(demand, info, station, years)
plt.show()

# Temperature-coded Christmas-only relative ranking
- coloured lines refer to the average temperature of that day for that year
- Using the 30+/- days around christmas

In [ ]:
def christmas_relative_rank_all_years_window_temp(demand, obs, info, station, years):
    """
    For each Christmas Day:
        • Build ±30-day window around Christmas
        • Compute relative ranks within that window
        • Extract the 24-hour Christmas Day relative-rank curve
        • Compute mean temperature for that Christmas Day from obs['t2m']
    Plot:
        • One line per Christmas Day
        • Line colour = mean temperature on that Christmas Day
    """

    # --- Ensure datetime index ---
    demand.index = pd.to_datetime(demand.index)
    obs.index = pd.to_datetime(obs.index)

    # Drop categorical column so resample().mean() works
    obs = obs.select_dtypes(include="number")

    hourly = demand[[station]].resample("h").mean()
    obs_hourly = obs.resample("h").mean()

    # --- Hourly demand + hourly temperature ---
    hourly = demand[[station]].resample("h").mean()
    obs_hourly = obs.resample("h").mean()   # contains t2m, t2m_30max, t2m_30min, t2m_bin

    christmas_curves = []
    christmas_temps = []
    christmas_labels = []

    for year in years:

        # --- Build ±30-day window ---
        ref_date = pd.Timestamp(f"{year}-12-25")
        start = ref_date - pd.Timedelta(days=30)
        end   = ref_date + pd.Timedelta(days=30)

        window = hourly.loc[start:end].copy()
        if window.empty:
            continue

        window["date"] = window.index.date
        window["hour"] = window.index.hour

        # --- Compute relative rank within this window ---
        window["rank"] = window.groupby("hour")[station].rank(method="average")
        n_days = window.groupby("hour")["date"].transform("nunique")
        window["relative_rank"] = window["rank"] / n_days

        # --- Extract Christmas Day relative-rank curve ---
        expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")
        christmas_day = window["relative_rank"].reindex(expected_hours)

        if christmas_day.isna().all():
            continue

        christmas_day.index = range(24)
        christmas_curves.append(christmas_day)
        christmas_labels.append(year)

        # --- Compute mean temperature for that Christmas Day ---
        # Only use t2m (numeric), avoids category dtype error from t2m_bin
        temp_day = obs_hourly["t2m"].reindex(expected_hours)
        mean_temp = temp_day.mean()
        christmas_temps.append(mean_temp)

    # --- Convert curves to DataFrame ---
    curves_df = pd.DataFrame(christmas_curves, index=christmas_labels)

    # --- Colour map based on temperature ---
    import matplotlib
    import numpy as np

    temps = np.array(christmas_temps)
    norm = matplotlib.colors.Normalize(vmin=temps.min(), vmax=temps.max())
    cmap = matplotlib.colormaps.get_cmap("coolwarm")

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(14,4))

    for i, (year, row) in enumerate(curves_df.iterrows()):
        temp = temps[i]
        color = cmap(norm(temp))

        ax.plot(
            row.index,
            row.values,
            color=color,
            linewidth=2,
            alpha=0.9,
            label=f"{year} ({temp:.1f}°C)"
        )

    # --- Formatting ---
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_yticklabels(["Lowest", "25%", "50%", "75%", "Highest"])

    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(f"{full_name} Christmas Relative Rank (±30d window)\nColoured by Mean Temperature")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Relative Rank (0 = lowest, 1 = highest)")

    # --- Colourbar ---
    sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Mean Christmas Day Temperature (°C)")

    ax.legend(
    ncol=4,
    fontsize=8,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.25)
    )
    return fig

In [ ]:
station = "BLAKE"
years = list(range(2004, 2018))

fig = christmas_relative_rank_all_years_window_temp(demand, obs, info, station, years)
plt.show()

# Temperature-coded relative ranking
- made for any public holiday

In [ ]:
def holiday_relative_rank_all_years_window_temp(
    demand, obs, info, station, years, holiday_name, holiday_lib, window_days=30
):
    """
    For each holiday:
        • Build ±window_days window around the holiday
        • Compute relative ranks within that window
        • Extract the 24-hour holiday relative-rank curve
        • Compute mean temperature for that holiday from obs['t2m']
    Plot:
        • One line per holiday occurrence
        • Line colour = mean temperature on that holiday
    """

    # --- Ensure datetime index ---
    demand.index = pd.to_datetime(demand.index)
    obs.index = pd.to_datetime(obs.index)

    # Drop categorical column so resample().mean() works
    obs = obs.select_dtypes(include="number")

    hourly = demand[[station]].resample("h").mean()
    obs_hourly = obs.resample("h").mean()

    curves = []
    temps = []
    labels = []

    for year in years:

        # --- Get holiday date from library ---
        if holiday_name not in holiday_lib:
            raise ValueError(f"Holiday '{holiday_name}' not found in holiday library")

        ref_date = holiday_lib[holiday_name](year)

        # --- Build ±window_days window ---
        start = ref_date - pd.Timedelta(days=window_days)
        end   = ref_date + pd.Timedelta(days=window_days)

        window = hourly.loc[start:end].copy()
        if window.empty:
            continue

        window["date"] = window.index.date
        window["hour"] = window.index.hour

        # --- Compute relative rank within this window ---
        window["rank"] = window.groupby("hour")[station].rank(method="average")
        n_days = window.groupby("hour")["date"].transform("nunique")
        window["relative_rank"] = window["rank"] / n_days

        # --- Extract holiday relative-rank curve ---
        expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")
        holiday_day = window["relative_rank"].reindex(expected_hours)

        if holiday_day.isna().all():
            continue

        holiday_day.index = range(24)
        curves.append(holiday_day)
        labels.append(year)

        # --- Compute mean temperature for that holiday ---
        temp_day = obs_hourly["t2m"].reindex(expected_hours)
        temps.append(temp_day.mean())

    # --- Convert curves to DataFrame ---
    curves_df = pd.DataFrame(curves, index=labels)

    # --- Colour map based on temperature ---
    import matplotlib
    import numpy as np

    temps = np.array(temps)
    norm = matplotlib.colors.Normalize(vmin=temps.min(), vmax=temps.max())
    cmap = matplotlib.colormaps.get_cmap("coolwarm")

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(14,4))

    for i, (year, row) in enumerate(curves_df.iterrows()):
        color = cmap(norm(temps[i]))
        ax.plot(
            row.index,
            row.values,
            color=color,
            linewidth=2,
            alpha=0.9,
            label=f"{year} ({temps[i]:.1f}°C)"
        )

    # --- Formatting ---
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_yticklabels(["0", "0.25", "0.5", "0.75", "1"])

    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(
        f"{full_name} {holiday_name} Relative Rank (±{window_days}d window)\n"
        "Coloured by Mean Temperature"
    )
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Relative Rank (0 = lowest, 1 = highest)")

    # --- Colourbar ---
    sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label(f"Mean {holiday_name} Temperature (°C)")

    ax.legend(
        ncol=4,
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.25)
    )

    return fig

In [ ]:
#looping and saving

def generate_all_relative_rank_plots(
    demand, obs, info, years, holiday_lib,
    base_dir="/home/565/pv3484/aus_substation_electricity/figures/relative_ranking"
):
    """
    Loops through:
        • each holiday in holiday_lib
        • each station in info.index
    Saves each figure into:
        base_dir/<HolidayName>/<STATION>.png
    """

    for holiday_name in holiday_lib.keys():

        # --- Create folder for this holiday ---
        holiday_folder = os.path.join(base_dir, holiday_name.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        print(f"\n=== {holiday_name} ===")

        for station in info.index:

            print(f"   → {station}")

            fig = holiday_relative_rank_all_years_window_temp(
                demand=demand,
                obs=obs,
                info=info,
                station=station,
                years=years,
                holiday_name=holiday_name,
                holiday_lib=holiday_lib,
                window_days=30
            )

            out_path = os.path.join(holiday_folder, f"{station}.png")
            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)


In [ ]:
years = list(range(2004, 2018))

generate_all_relative_rank_plots(
    demand=demand,
    obs=obs,
    info=info,
    years=years,
    holiday_lib=HOLIDAYS_VIC
)